# 3장 2강: 규제 — L1(Lasso), L2(Ridge), ElasticNet

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 Ridge, Lasso, ElasticNet 규제가 회귀계수와 일반화 성능에 어떤 영향을 주는지 확인합니다.

- 규제가 계수의 크기를 줄이는 이유를 이해합니다.
- `alpha` 변화에 따라 Ridge의 계수 크기와 Test 성능이 어떻게 달라지는지 확인합니다.
- Lasso가 일부 계수를 정확히 0으로 만드는지 확인합니다.
- ElasticNet이 L1과 L2를 함께 사용하는 모델이라는 점을 확인합니다.
- Ridge / Lasso / ElasticNet의 계수와 Test 성능을 비교합니다.

> 이 실습에서는 3장 2강 교안에서 다룬 규제, alpha, Lasso, Ridge, ElasticNet, StandardScaler, R² 비교만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- numpy
- matplotlib
- scikit-learn
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

규제에 따른 계수 변화를 비교하기 위해 결측치가 없는 숫자형 특성 중 다음 12개를 사용합니다.

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

규제는 계수의 크기를 기준으로 벌점을 주기 때문에 특성의 스케일을 맞추는 과정이 중요합니다.

이번 실습에서는 다음 순서로 데이터를 준비합니다.

1. Ames Housing 데이터 불러오기
2. 사용할 특성과 `SalePrice` 선택
3. Train / Test 분리
4. `StandardScaler`를 이용한 특성 표준화

In [1]:
# ===== 셀 1: 데이터 준비 · 분할 · 스케일링 =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
Y = house["SalePrice"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # train으로 평균·표준편차를 구하고(fit) 변환
X_test_scaled = scaler.transform(X_test)         # test는 train의 기준으로 변환만(transform)

print("Train 크기:", X_train_scaled.shape)
print("Test 크기:", X_test_scaled.shape)


Train 크기: (1168, 12)
Test 크기: (292, 12)


---

# 필수 1. Ridge와 alpha에 따른 계수 축소

## 배경

Ridge는 **L2 규제**를 사용합니다.

Ridge는 계수의 제곱합에 벌점을 주기 때문에 큰 계수를 더 강하게 억제하고, 모든 계수를 0에 가깝게 줄이는 특징이 있습니다.

## 문제 1. 여러 alpha 값에서 Ridge를 비교하세요.

### 요구사항

다음 alpha 값을 사용합니다.

```python
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]
```

각 alpha에 대해 다음을 수행합니다.

1. `Ridge(alpha=alpha)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test 데이터를 예측합니다.
4. Test R²를 계산합니다.
5. `abs(model.coef_).sum()`으로 전체 계수 크기를 확인합니다.
6. 결과를 DataFrame으로 정리합니다.

### 확인 포인트

- alpha가 커질수록 규제 강도가 강해지는가?
- alpha가 커질수록 전체 계수 크기가 작아지는가?
- alpha가 너무 커지면 Test 성능이 낮아질 수 있는가?

In [2]:
X_train.mean().round(2)
X_train.std().round(2)

OverallQual       1.37
GrLivArea       524.43
GarageCars        0.74
GarageArea      211.10
TotalBsmtSF     440.68
1stFlrSF        386.68
2ndFlrSF        439.57
FullBath          0.55
TotRmsAbvGrd      1.62
YearBuilt        30.68
YearRemodAdd     20.73
Fireplaces        0.65
dtype: float64

In [3]:

# ===== 셀 2: 여러 alpha 값에서 Ridge 비교 =====
# TODO
# 여러 alpha 값에 대해 Ridge의 Test R²와 계수 크기를 비교하세요.

alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_results = []

for alpha in alpha_values:
    model = Ridge(alpha = alpha)
    model.fit(X_train_scaled, Y_train)
    y_pred = model.predict(X_test_scaled)
    test_r2 = r2_score(Y_test, y_pred)
    coefficient_size = abs(model.coef_).sum()
    ridge_results.append({"alpha": alpha, "test_r2": test_r2, "coefficient_size": coefficient_size})
ridge_results_df = pd.DataFrame(ridge_results)
display(ridge_results_df)

,alpha,test_r2,coefficient_size
0,0.001,0.799357,102172.590850
1,0.010,0.799357,102172.463666
2,0.100,0.799355,102171.096834
3,1.000,0.799331,102149.831747
4,10.000,0.799056,101725.431454
5,100.000,0.795593,97390.120153


 
|alpha | Test R2 | 계수 절대값 합 | 내용 |
|------|---------|--------------|-----|
|0.001 |0.799357|102172|규제가 가장 약한 설정의 값|
|0.01|0.799357|102172| 앞서 규제가 가장 약했던 설정의 값과 계수가 일치|
|0.1|0.799355|102171| 규제는 지속적으로 커졌지만 여전히 가장 약했던 설정의 계수와 거의 일치|
|1|0.799331|102149|계수 합이 조금 감소했으며 R2값이 낮아짐|
|10|0.799056|101725|계수 축수가 뚜렷하며, R2가 낮아짐|
|100|0.795593|97390|이번 후보중 계수의 합이 가장 낮으며 R2값도 가장 낮음|

# Q1. alpha가 커질수록 Ridge 계수는 일반적으로 어떻게 변하나요?
: 규제를 강하게 할수록 Ridge계수가 어떻게 변화느냐, 일반적으로 계수의 절대값이 작아진다.
: Ridge는 모든 계수를 한번에 0으로 바꾸지 않고 0에 가까운 방향으로 축소하려고 한다. 

# Q2. alpha가 너무 크면 어떤 문제가 발생할 수 있나요?
: 특성화 계수를 굉장히 낮춰서 과소적합이 발생할 수 있다. 중요한 특성의 계수를 지나치게 줄여서 학습에 충분한 힌트를 얻지 못할 수 있다.
: 즉, 과소적합이 발생할 수 있다. 

---

# 필수 2. Lasso와 ElasticNet의 계수 변화

## 문제 1. Lasso에서 0이 되는 계수를 확인하세요.

### 배경

Lasso는 **L1 규제**를 사용하며, 중요하지 않은 특성의 계수를 정확히 0으로 만들 수 있습니다.

이번에는 규제 강도를 충분히 높여 Lasso의 변수 선택 효과를 직접 확인합니다.

### 요구사항

1. `Lasso(alpha=1000, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. 각 특성의 계수를 DataFrame으로 만듭니다.
4. 계수가 0인 특성의 개수를 계산합니다.
5. Test R²를 계산합니다.

### 확인 포인트

- 일부 계수가 정확히 0이 되는가?
- Ridge와 달리 Lasso가 변수 선택 효과를 갖는 이유를 이해했는가?

In [4]:
# TODO
# Lasso(alpha=1000)를 학습하고 계수와 0 계수 개수를 확인하세요.

lasso_model = Lasso(alpha=1000, max_iter = 10000)
lasso_model.fit(X_train_scaled, Y_train)
lasso_pred = lasso_model.predict(X_test_scaled)

lasso_r2 = r2_score(Y_test, lasso_pred)

lasso_coef_df = pd.DataFrame({"feature": features, "coefficient": lasso_model.coef_})

display(lasso_coef_df)
zero_count = (lasso_model.coef_== 0).sum()

print("0이 된 계수 개수", zero_count)
print("Test R²:" , lasso_r2)



,feature,coefficient
0,OverallQual,25520.693859
1,GrLivArea,21419.631042
2,GarageCars,6820.209072
3,GarageArea,4370.090292
4,TotalBsmtSF,7305.573442
5,1stFlrSF,3521.096689
6,2ndFlrSF,0.000000
7,FullBath,-0.000000
8,TotRmsAbvGrd,83.200163
9,YearBuilt,6966.923170


0이 된 계수 개수 2
Test R²: 0.7960182963880562


In [5]:
# *계수를 자세히 볼 필요. * 
# 결과 
# OverallQual 컬럼의 계수 25520 의 의미: 품질이 높을수록 예측 가격이 높아지는 방향, 그 강도
# GrLivArea (차고)컬럼의 계수 21419의 의미: 지상 면적의 정보도 예측 가격에 양의 계수로 반영되고 있다.
# 2ndF1rSF(1층면적)컬럼의 계수 0.0의 의미: 이 모델에서 이 2층 면적의 직접적인 기여도는 0이다.
# FullBath(욕실)컬럼의 계수 -0.0의 의미: 이 모델에서 욕실은 예측 가격에 직접적인 기여도가 0이다. 

# 0인 계수가 2개인 것의 의미
# 전체 12개의 특성중 2개의 특성이 0의 특징을 보임.
# 따라서 이 두열의 값이 바뀌거나 제거되어도 나머지 10개의 얼이 같다면 모델의 결과는 달라지지 않음.

# 계수0 은 그러면 모델링할 때 삭제해도 무방한가?
# -> 2층 면적이나 욕실수는 계수가 없어서 주택 가격과 관련이 없다는 것이 아닌가?
# -> 앞에 조건이 붙음, ->나머지 10의 컬럼이 같다면->나머지 10개의 컬럼이 달라진다면, 값이 바뀔 수 있다.
# -> 데이터 분할, 규제 강도의 변화, x의 변화가 있는 경우 계수가 0이 아닐 수도 있음. 

# # Q3. Lasso에서 계수가 0이 된다는 것은 어떤 의미인가요?
# : 해당 특성(x)가 최종 예측식에서 영향을 주지 않는다는 의미
# : Lasso의 특징은 중요도가 낮다고 판단한 특성을 자동으로 모델의 힌트로 사용하지 않음.(제거하는 효과).

## 문제 2. ElasticNet을 학습하고 L1/L2 혼합을 확인하세요.
-절댓갑과 제곱근을 같이 사용해서 훨씬 더 강한 규제를 적용하겠다는 의미. 
### 배경

ElasticNet은 L1과 L2 규제를 함께 사용합니다.

`l1_ratio`는 두 규제의 비율을 조절합니다.

- `l1_ratio`가 1에 가까움 → Lasso 성격이 강함
- `l1_ratio`가 0에 가까움 → Ridge 성격이 강함

### 요구사항

1. `ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test R²를 계산합니다.
4. 각 특성의 계수를 확인합니다.

### 확인 포인트

- `alpha`와 `l1_ratio`의 역할을 구분할 수 있는가?
- ElasticNet이 Lasso와 Ridge의 성격을 함께 가진다는 점을 설명할 수 있는가?

In [6]:
# TODO
# ElasticNet(alpha=0.1, l1_ratio=0.5)을 학습하세요.

# 1. 모델 만들기
#    alpha=0.1      : 규제의 전체 세기
#    l1_ratio=0.5   : L1(Lasso)과 L2(Ridge)를 반반 섞음
#    max_iter=10000 : 계수를 찾는 반복 계산의 최대 횟수
elastic_model = ElasticNet(alpha = 0.1, l1_ratio = 0.5, max_iter = 10000)
 
# 2. Train 데이터로 학습
elastic_model.fit(X_train_scaled, Y_train)
 
# 3. Test 데이터를 예측하고 Test R² 계산
elastic_pred = elastic_model.predict(X_test_scaled)
elastic_r2 = r2_score(Y_test, elastic_pred)
print(f"ElasticNet Test R²: {elastic_r2:.6f}")
 
# 4. 각 특성의 계수 확인 (절댓값이 큰 순서로 정렬)
elastic_coef_df = pd.DataFrame({"feature": features, "coefficient": elastic_model.coef_})
elastic_coef_df = elastic_coef_df.sort_values("coefficient", key=abs, ascending=False)
display(elastic_coef_df)
 
# 0이 된 계수 개수 (Lasso처럼 계수가 0이 되었는지 확인)
print("0이 된 계수 개수:", (elastic_model.coef_ == 0).sum())

ElasticNet Test R²: 0.797291


,feature,coefficient
0,OverallQual,23514.629015
1,GrLivArea,12670.768121
5,1stFlrSF,9237.815373
9,YearBuilt,8072.285509
4,TotalBsmtSF,8016.089858
10,YearRemodAdd,7518.255622
11,Fireplaces,6995.256241
6,2ndFlrSF,6873.818086
2,GarageCars,6785.127803
3,GarageArea,5137.437024


0이 된 계수 개수: 0


# 1_ratio = 0.5 : L1 항의 앞에 0.05, L2제곱합 앞에는 0.025, r=0.5라는 의미는 두 규제의 혼합비율을 중간으로 쓰겠다 라는 의미.

결과. 
OverallQual(품질) 컬럼의 계수 23514의 의미: 양의 계수중 가장 큰 값, 품질 정보가 가격예측에 가장 크게 반영.
GrLivArea(지상면적) 컬럼의 계수 12670의 의미: 양의 계수중 두번째로 큰 값, 면적도 가격예측에 크게 반영. 
2ndFlSF(2층면적) 컬럼의 계수 6873의 의미: Lasso에서는 0이 였는데 현재에서는 가격예측에 기역(하고 있는 컬럼)
FullBath (욕실) 컬럼의 계수 -1895의 의미: 다른 입력이 같다면 가격예측에 음의 영향을 보인다. 


# Q4. `alpha`와 `l1_ratio`는 각각 무엇을 조절하나요?
alpha : 전체 규제의 강도를 조절, 
L1_ratio : 규제 안에서 L1 과 L2 를 어떤 비율로 사용할 지 조절
-> L1_ration가 1에 가까워질수록  Lasso, 0에 가까워질수록 Ridge에 성향이 강해진다.



---

# 과제 1. 다중공선성 진단과 Ridge / Lasso / ElasticNet 비교

## 배경

같은 Train/Test 데이터에서 세 규제 모델을 비교합니다. 성취도 평가항목에 맞춰, 비교 전에 **다중공선성의 계수 해석 영향과 VIF 진단·처치 선택**을 추가합니다.

다중공선성은 입력 특성들이 서로 강한 선형 관계를 갖는 상황입니다. VIF는 한 특성을 나머지 특성으로 설명할 때 구한 결정계수 R²를 이용해 `1 / (1 - R²)`로 계산합니다. 이때 예측 대상은 `SalePrice`가 아니라 **진단할 입력 특성 하나**이며, 보조 회귀에는 절편을 포함합니다.

## 요구사항

### A. 다중공선성 진단과 처치 방향 선택

1. **Train의 입력 특성만** 이용해 특성별 VIF를 계산하고 내림차순 표 `vif_df`를 출력합니다. 기존 `X_train_scaled`를 사용해도 됩니다. 표준화는 특성 간 선형 관계 자체를 제거하지 않습니다.
2. 이번 과제에서는 `VIF < 5`: 상대적으로 낮음, `5 ≤ VIF < 10`: 주의, `VIF ≥ 10`: 높은 수준으로 분류합니다. 이는 절대적 삭제 규칙이 아닌 실습용 판단 기준입니다.
3. 주의·높은 수준인 특성을 특성명과 VIF로 제시합니다. 없으면 없다고 작성합니다. 완전한 선형 중복으로 VIF가 무한대인 경우 높은 수준으로 해석합니다. 상수 특성의 VIF는 정의하기 어려우므로 별도로 표시합니다.
4. 다중공선성이 계수의 크기·부호·안정성과 개별 변수 해석에 미치는 영향을 설명합니다.
5. 특성의 의미와 VIF 결과를 근거로 **변수 제거·결합·규제 중 한 가지 처치 방향**을 선택하고 대상·이유·한계를 설명합니다. 높은 VIF가 없으면 불필요한 제거를 하지 않는 이유와 기존 규제 비교의 목적을 설명해도 됩니다. 제거·결합의 추가 구현은 요구하지 않습니다.

### B. 기존 규제 모델 비교

다음 설정을 그대로 사용합니다.

```python
Ridge(alpha=1.0)
Lasso(alpha=1000, max_iter=10000)
ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
```

1. 세 모델을 동일한 표준화 Train 데이터로 학습합니다.
2. 동일한 Test 데이터의 R²를 계산합니다.
3. 전체 계수 크기 `abs(coef_).sum()`과 계수가 정확히 0인 개수를 계산합니다.
4. 결과를 `comparison_df`로 정리하고, 특성별 계수를 `task_coef_df`로 출력합니다.
5. Test R²와 계수 처리 차이를 함께 설명하고 Q5~Q9에 답합니다.

> Test 점수는 지정된 세 설정의 결과를 비교·해석하는 데 사용합니다. 이 점수를 보고 변수를 제거하거나 alpha를 반복 조정하지 않습니다. 실제 모델 선택·튜닝에는 Train 내부 검증 또는 CV가 필요합니다.

### 확인 포인트

- Train 입력 특성만 사용하여 VIF를 계산했는가?
- 기준과 실제 수치를 근거로 주의할 특성을 판단했는가?
- 다중공선성의 계수 해석 영향과 처치 선택 이유를 설명했는가?
- 규제가 입력 특성의 VIF 자체를 낮추는 것은 아님을 구분했는가?
- 같은 데이터에서 세 모델의 Test R²·계수 크기·0 계수 개수를 비교했는가?


In [8]:
# TODO
# A. Train 특성별 VIF 계산 및 판단
import numpy as np
import pandas as pd

from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = pd.DataFrame(X_train_scaled, columns=features)

vif_values = []

for i in range(X_vif.shape[1]):
    if X_vif.iloc[:, i].nunique() <= 1:
        vif_values.append(np.nan)
    else:
        vif = variance_inflation_factor(X_vif.values, i)
        vif_values.append(vif)

vif_df = pd.DataFrame({
    "feature": features,
    "VIF": vif_values
})

vif_df = vif_df.sort_values("VIF", ascending=False)

display(vif_df)

#  VIF 수준 분류
def vif_level(vif):
    if pd.isna(vif):
        return "상수 특성"
    elif vif >= 10:
        return "높음"
    elif vif >= 5:
        return "주의"
    else:
        return "낮음"

vif_df["level"] = vif_df["VIF"].apply(vif_level)

display(vif_df)

#  3. 주의 및 높은 수준의 특성 확인
high_vif_df = vif_df[vif_df["VIF"] >= 5]

print("VIF가 5 이상인 특성:")

if len(high_vif_df) == 0:
    print("없음")
else:
    display(high_vif_df[["feature", "VIF", "level"]])

print("상수 특성:")
constant_vif_df = vif_df[vif_df["level"] == "상수 특성"]

if len(constant_vif_df) == 0:
    print("없음")
else:
    display(constant_vif_df)

# B. 기존 세 규제 모델 비교 및 특성별 계수 표 출력
# Ridge / Lasso / ElasticNet 비교
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score

# TODO 4. 모델 생성
ridge_model = Ridge(alpha=1.0)
lasso_model = Lasso(alpha=1000, max_iter=10000)
elastic_model = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)

# Train 데이터 학습
ridge_model.fit(X_train_scaled, Y_train)
lasso_model.fit(X_train_scaled, Y_train)
elastic_model.fit(X_train_scaled, Y_train)

# Q5~Q9 답변 작성
# Test R² 계산
# TODO 5. Test 예측
ridge_pred = ridge_model.predict(X_test_scaled)
lasso_pred = lasso_model.predict(X_test_scaled)
elastic_pred = elastic_model.predict(X_test_scaled)

# Test R² 계산
ridge_r2 = r2_score(Y_test, ridge_pred)
lasso_r2 = r2_score(Y_test, lasso_pred)
elastic_r2 = r2_score(Y_test, elastic_pred)

print(f"Ridge Test R²: {ridge_r2:.6f}")
print(f"Lasso Test R²: {lasso_r2:.6f}")
print(f"ElasticNet Test R²: {elastic_r2:.6f}")

# 6. 전체 계수 크기와 0인 계수 개수 계산
# TODO 6. 계수 절댓값의 합
ridge_coef_sum = abs(ridge_model.coef_).sum()
lasso_coef_sum = abs(lasso_model.coef_).sum()
elastic_coef_sum = abs(elastic_model.coef_).sum()

# 0인 계수 개수
ridge_zero = (ridge_model.coef_ == 0).sum()
lasso_zero = (lasso_model.coef_ == 0).sum()
elastic_zero = (elastic_model.coef_ == 0).sum()

# 7. 세 모델 비교표 생성
comparison_df = pd.DataFrame({
    "model": ["Ridge", "Lasso", "ElasticNet"],
    "test_r2": [ridge_r2, lasso_r2, elastic_r2],
    "coef_abs_sum": [ridge_coef_sum, lasso_coef_sum, elastic_coef_sum],
    "zero_coef_count": [ridge_zero, lasso_zero, elastic_zero]
})

display(comparison_df)

#  8. 특성별 계수 비교
task_coef_df = pd.DataFrame({
    "feature": features,
    "Ridge": ridge_model.coef_,
    "Lasso": lasso_model.coef_,
    "ElasticNet": elastic_model.coef_
})

display(task_coef_df)

,feature,VIF
1,GrLivArea,128.776832
6,2ndFlrSF,91.620664
5,1stFlrSF,73.585999
2,GarageCars,5.290892
3,GarageArea,5.032498
4,TotalBsmtSF,3.792935
8,TotRmsAbvGrd,3.333240
0,OverallQual,2.688595
9,YearBuilt,2.389316
7,FullBath,2.234330


,feature,VIF,level
1,GrLivArea,128.776832,높음
6,2ndFlrSF,91.620664,높음
5,1stFlrSF,73.585999,높음
2,GarageCars,5.290892,주의
3,GarageArea,5.032498,주의
4,TotalBsmtSF,3.792935,낮음
8,TotRmsAbvGrd,3.333240,낮음
0,OverallQual,2.688595,낮음
9,YearBuilt,2.389316,낮음
7,FullBath,2.234330,낮음


VIF가 5 이상인 특성:


,feature,VIF,level
1,GrLivArea,128.776832,높음
6,2ndFlrSF,91.620664,높음
5,1stFlrSF,73.585999,높음
2,GarageCars,5.290892,주의
3,GarageArea,5.032498,주의


상수 특성:
없음
Ridge Test R²: 0.799331
Lasso Test R²: 0.796018
ElasticNet Test R²: 0.797291


,model,test_r2,coef_abs_sum,zero_coef_count
0,Ridge,0.799331,102149.831747,0
1,Lasso,0.796018,88648.006135,2
2,ElasticNet,0.797291,99258.844252,0


,feature,Ridge,Lasso,ElasticNet
0,OverallQual,25310.894717,25520.693859,23514.629015
1,GrLivArea,13960.707641,21419.631042,12670.768121
2,GarageCars,7041.543409,6820.209072,6785.127803
3,GarageArea,4382.957511,4370.090292,5137.437024
4,TotalBsmtSF,7238.590424,7305.573442,8016.089858
5,1stFlrSF,10011.019199,3521.096689,9237.815373
6,2ndFlrSF,7098.896701,0.000000,6873.818086
7,FullBath,-3154.791944,-0.000000,-1895.877032
8,TotRmsAbvGrd,1602.125673,83.200163,2541.484569
9,YearBuilt,8343.117317,6966.923170,8072.285509


### Q5. Ridge와 Lasso의 가장 큰 계수 처리 차이는 무엇인가요?

**답변:**
Ridge는 계수의 제곱에 벌점을 주어 계수 크기를 줄이는 방향으로 계수를 유지하지만, Lasso는 계수의 절댓값에 벌점을 주어서 일부 계수를 0으로 만들어 계수를 조절 및 변수를 선택할 수 있다. 


### Q6. 규제 모델의 성능을 비교할 때 Test R²뿐 아니라 계수도 함께 보는 이유는 무엇인가요?

**답변:**
계수는 각 특성이 예측값에 어떤 방향과 크기로 영향을 주었는지 확인 할 수 있기 때문에 규제 효과를 볼 수 있다. 예측 성능이 비슷하더라도 각 모델이 특성을 활용하는 방식이 다를 수 있기 떄문에 모델의 별 규제 성능을 비교하려면 계수를 함께 보아야 한다. 


### Q7. 다중공선성은 회귀계수의 크기·부호·안정성과 개별 변수의 영향 해석에 어떤 문제를 일으킬 수 있나요?

**답변:**
다중공선성은 입력 개별 변수들 사이의 강한 성형 관계가 존재하는 것으로 회귀계수의 크기.부호.안정성에 영향을 줄 수 있어 이는 개별 변수(특성)이 예측값에 미치는 영향을 해석하기 어렵게 한다. 


### Q8. VIF 판단 기준은 무엇이며, 계산 결과에서 주의하거나 처치를 검토할 특성은 무엇인가요? 특성명과 VIF를 근거로 답하세요.

**답변:**
VIF 5이상 10미만은 변수 간 관계 확인이 필요하고, 
10 이상인 경우 변수 제거.결합.규제를 검토해봐야 한다. 
세 변수는 면적에 관한 변수로 다중공선성이 높다고 볼 수 있다. 때문에 VIF가 10미만이더라도 처치를 검토할 필요가 있다. 
GrLivArea	128.776832	높음
2ndFlrSF	91.620664	높음
1stFlrSF	73.585999	높음


### Q9. VIF 결과와 특성의 의미를 고려하여 변수 제거·결합·규제 중 어떤 처치 방법을 선택하겠나요? 구체적인 대상, 선택 이유와 한계를 설명하세요.

**답변:**
규제, VIF결과에서 GrLivArea, 2ndFlrSF, 1stFlrSF 모두 높게 나타났기 때문이다. 결합이나 제거는 면적 관련 각 변수들의 의미를 유지하고, GrLivArea 는 2ndFlrSF, 1stFlrSF 을 포함하는 변수이기 때문에 결합을 하지 않았따. 

---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 규제는 손실함수에 무엇을 추가하는 방법인가요?
: 손실함수에 벌점(계수에 제약을 거는 것이다.)을 주는 것.
: 숫자가 지속적으로 커지는 것을 제한하겠다. 영향력을 줄이겠다 라는 것.
-> 손실함수에 계수 크기에 대한 벌점을 추가한다. 

2. alpha가 커지면 규제 강도는 어떻게 변하나요?
-> 규제 강도가 강해진다. 

3. Ridge의 계수는 일반적으로 어떻게 변하나요?
-> 모든 계수를 0에 가까운 방향으로 축소함, 단0으로 만들지는 않음. 

4. Lasso가 변수 선택 효과를 가지는 이유는 무엇인가요?
-> 일부 계수를 0으로 만들 수 있다. 

5. ElasticNet의 `l1_ratio=0.5`는 어떤 의미인가요?
-> 0.5의 의미는 L1과 L2를 절반씩 사용하겠다.L1, L2규제를 절반씩 섞는 설정으로 이해할 수 있다. 

6. 규제 전에 특성 스케일을 맞추는 이유는 무엇인가요?
-> 규제는 계수 크기를 기준으로 패널티를 부여하기 때문에 특성들의 값의 범위가다르면 패널티가 다르게 반영될 수 있다.
-> 따라서 특성 스케일을 맞춘 뒤 규제를 적용하였다. 

--특성 차이 dicision tress 특성 스케일링에 대한 설명--보고서 부분. 데이터 불균형시 모델링이 어떻게 나오고 어떻게 계산할지 보는 것. 